In [ ]:
from preamble_jax import *

# Read Data
## Save as Rainbows

In [ ]:
G102_extracted_spec = '../../data/PACMAN/G102/run_2024-10-19_12-00-36_AUMic_G102/extracted_lc/2024-10-19_12-05-03/lc_spec.txt'
G141_extracted_spec = '../../data/PACMAN/G141/run_2024-10-19_12-11-10_AUMic_G141/extracted_lc/2024-10-19_12-30-32/lc_spec.txt'

In [ ]:
# Open the .txt file using astropy's Table.read
F21_table = Table.read(G141_extracted_spec, format='ascii')
S22_table = Table.read(G102_extracted_spec, format='ascii')

# Split the dataset based on scan value
F21_forward_scan = F21_table[F21_table['scan'] == 0.0]
F21_reverse_scan = F21_table[F21_table['scan'] == 1.0]
S22_forward_scan = S22_table[S22_table['scan'] == 0.0]
S22_reverse_scan = S22_table[S22_table['scan'] == 1.0]

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(6,3))

for visit in ['F21','S22']:    
    for direction in ['Forward','Reverse']:
        
        if visit=='F21':
            if direction =='Forward':
                table = F21_forward_scan
                col='darkred'
            if direction =='Reverse':        
                table = F21_reverse_scan
                col='pink'
        
        if visit=='S22':
            if direction =='Forward':
                table = S22_forward_scan
                col='darkred'
            if direction =='Reverse':  
                table = S22_reverse_scan
                col='pink'
                
        wavelength = table['template_waves'].data/1e4 * u.micron  # Wavelength
        flux = table['spec_opt'].data * u.electron      # Spectrum
        var = table['var_opt'].data     # Variance
        err = np.sqrt(var) * u.electron
        bjd_times = table['t_bjd'].data * u.day       # Times converted from MJD
        
        unique_times = np.unique(bjd_times)
        unique_wavelengths = np.unique(wavelength)
        
        flux_list = [None]*len(unique_times)
        err_list = [None]*len(unique_times)
        wave_list = [None]*len(unique_times)
        
        i = 0
        for t_i in unique_times:
            
            this_times_wavelengths = wavelength[bjd_times == t_i]
            this_times_fluxes = flux[bjd_times == t_i]
            this_times_errors = err[bjd_times == t_i]
            
            wave_list[i] = this_times_wavelengths
            flux_list[i] = this_times_fluxes
            err_list[i] = this_times_errors
                
            ax.errorbar(this_times_wavelengths, this_times_fluxes, yerr=this_times_errors, alpha=0.005,color=col)
            i += 1

        stacked_flux = np.stack(flux_list, axis = 1)
        stacked_err = np.stack(err_list, axis = 1)
        stacked_wave = np.stack(wave_list, axis = 1)
        
        pacman_rainbow = Rainbow(wavelength = np.nanmedian(stacked_wave, axis=1),
                          time = unique_times,
                          flux = stacked_flux,
                          uncertainty = stacked_err)
        # Save the unaltered extracted spec
        pacman_rainbow.save(f'../../data/rainbows/{visit}_{direction}_unaltered_pacman_spec.rainbow.npy')

        # In the event that the wavelength arrays are not identical, this will correct that
        pacman_rainbow.fluxlike["wavelength_2d"] = stacked_wave
        aligned = pacman_rainbow.align_wavelengths()      
        aligned.save(f'../../data/rainbows/{visit}_{direction}_aligned_pacman_spec.rainbow.npy')
        print(f'Successfully saved Rainbow object for {visit} {direction}')
        
fig.suptitle(f'Spectra extracted by PACMAN')
# ax.set_title('S22')
# ax.set_title('F21')
# plt.yscale('log')
plt.ylabel(r'Flux (e$^-$)')
plt.xlabel(r'Wavelength ($\mu$m)')
plt.savefig(f'../../figs/{visit}_pacman_spectra.png',dpi=300)
plt.show()

# Examine the spectroscopic light curves we just saved with Chromatic

In [ ]:
for visit in ['F21','S22']:

    if visit == 'F21':
        time_offset = 2459455 * u.day
        title = 'F21/G141'
    elif visit == 'S22':
        time_offset = 2459684 * u.day
        title = 'S22/G102'

    for direction in ['Forward','Reverse']:
                
        unaltered_rainbow = read_rainbow(f'../../data/rainbows/{visit}_{direction}_unaltered_pacman_spec.rainbow.npy')
        # unaltered_rainbow.pcolormesh()
        # plt.title(f'{visit} {direction} Unaligned Wavelengths')
        # plt.savefig(f'../../figs/{visit}_{direction}_pacman_unaligned_imshow.png',dpi=600)
        # plt.show()
        # plt.clf()

        # aligned_rainbow = read_rainbow(f'../../data/rainbows/{visit}_{direction}_aligned_pacman_spec.rainbow.npy')
        unaltered_rainbow.flux /= 1e6
        unaltered_rainbow.time = unaltered_rainbow.time - time_offset
        # unaltered_rainbow.pcolormesh()

        mesh = unaltered_rainbow.pcolormesh()  # This likely returns the colorbar
        # cbar = mesh.colorbar  # or mesh might be the mappable, and colorbar is attached

        # cbar.ax.tick_params(labelsize=13)  # Colorbar tick labels size
        # cbar.set_label('Flux (10⁻⁶)' , fontsize=13)  # Colorbar label with size

        plt.xlabel('')
        plt.tick_params(axis='both', labelsize=13)
        plt.ylabel(r'Wavelength ($\mu$m)', fontsize=13)
        plt.title(f'{title}', fontsize=16)

        plt.savefig(f'../../figs/{visit}_{direction}_pacman_aligned_imshow.png',dpi=600)
        plt.show()
        plt.clf()

In [ ]:
# for visit in ['F21','S22']:
    
#     for direction in ['Forward','Reverse']:

#         plt.figure(figsize=(7,3))
        
#         unaltered_whitelight_flux = np.nansum(unaltered_rainbow.flux, axis = 0)
#         plt.title(f'{visit} {direction} Unaligned Wavelengths')
#         plt.scatter(unaltered_rainbow.time, unaltered_whitelight_flux,label='Uncorrected Wavelengths',
#                     color='gray',s=2)

#         aligned_whitelight_flux = np.nansum(aligned_rainbow.flux, axis = 0)
#         aligned_whitelight_err = np.nansum(aligned_rainbow.uncertainty, axis = 0)
#         plt.title(f'{visit} {direction} Aligned Wavelengths')
#         plt.scatter(aligned_rainbow.time, aligned_whitelight_flux, label='Corrected Wavelengths',
#                     color='black',s=2)

#         plt.legend()
#         plt.savefig(f'../../figs/{visit}_{direction}_pacman_whitelightcurve.png',dpi=600)
#         plt.show()        

# Trim the edges of each spectrum, save as a new object

In [ ]:
for visit in ['F21','S22']:
    
    fig, (ax1, ax2) = plt.subplots(1,2,figsize=(12,4))

    for direction in ['Forward','Reverse']:
        print(visit,direction)
        print('')

        'Read in the data from the previous step'
        aligned_rainbow = read_rainbow(f'../../data/rainbows/{visit}_{direction}_aligned_pacman_spec.rainbow.npy')

        'Calculate the average spectrum'
        average_spec = aligned_rainbow.get_average_spectrum()
        average_spec_err = 0.005*average_spec

        visit_data = visits[f'{visit}']        
        grism = visit_data['Grism']
        t0 = visit_data['T0 (BJD_TDB)']
        if visit =='F21':
            wave_lower = 1.13*u.micron # blue-end cutoff for trimming
            wave_upper = 1.64*u.micron # red-end cutoff for trimming
            grism = 'G141'

        if visit =='S22':
            wave_lower = 0.79*u.micron # blue-end cutoff for trimming
            wave_upper = 1.13*u.micron # red-end cutoff for trimming
            grism = 'G102'
        
        'Calculate the white light curve'
        white_light_curve = np.nansum(aligned_rainbow.flux, axis=0)
        white_light_curve_err = 30e-6*white_light_curve
        
        'Plot'
        ax1.errorbar(aligned_rainbow.wavelength, average_spec/1e6, yerr=average_spec_err/1e6,label=f'{direction}')
        ax2.scatter((aligned_rainbow.time.value-t0.value)*24, white_light_curve/np.nanmean(white_light_curve[22:]), s=3,alpha=0.5, label=f'{direction}')

    # ax1.axhline(0.25*np.nanmax(average_spec.value),label=r'25$\%$ Peak Fluence')
    ax1.axvline(wave_lower.value,label=f'{wave_lower.value} micron',color='blue')
    ax1.axvline(wave_upper.value,label=f'{wave_upper.value} micron',color='red')
    ax1.legend(fontsize=12)
    ax1.set_title(f'Time-averaged spectrum',fontsize=15)
    ax1.set_xlabel(r'Wavelength [$\mu$m]',fontsize=14)
    ax1.set_ylabel(r'Flux [$\times10^6$ e-]',fontsize=14)
    
    ax2.axvspan(-1.75, 1.75, alpha=0.3,color='r',label='Transit of AU Mic b')
    if visit =='S22':
        ax2.axvspan(-3.5, -3.25, alpha=0.3,color='g',label='Flare')
    # ax2.axvspan((first_orbit_start.value-t0.value)*24, (first_orbit_end.value-t0.value)*24,label=f'First Orbit',alpha=0.3,zorder=-100,color='gray',linestyle='')
    ax2.legend(fontsize=12)
    ax2.set_title(f'Bandpass-Integrated Light Curve')
    ax2.set_xlabel(r'Time from transit (hr)',fontsize=14)
    ax2.set_ylabel('Relative Flux',fontsize=14)

    ax1.tick_params(axis='both',labelsize=12)
    ax2.tick_params(axis='both',labelsize=12)
    
    plt.suptitle(f'{visit}/{grism} Untrimmed Data',size=20)
    plt.savefig(f"../../figs/{visit}_untrimmed_pacman_data.png",dpi=300)

In [ ]:
for visit in ['F21','S22']:
    
    # fig, (ax1, ax2) = plt.subplots(1,2,figsize=(12,4))

    for direction in ['Forward','Reverse']:
        print(visit,direction)
        print('')
        
        'Load data tables'
        visit_data = visits[f'{visit}']        
        t0 = visit_data['T0 (BJD_TDB)']
       
        if visit =='F21':
            wave_lower = 1.13*u.micron # blue-end cutoff for trimming
            wave_upper = 1.64*u.micron # red-end cutoff for trimming
            grism = 'G141'

        if visit =='S22':
            wave_lower = 0.79*u.micron # blue-end cutoff for trimming
            wave_upper = 1.13*u.micron # red-end cutoff for trimming
            grism = 'G102'
            
        aligned_rainbow = read_rainbow(f'../../data/rainbows/{visit}_{direction}_aligned_pacman_spec.rainbow.npy')
        wavelength = aligned_rainbow.wavelength
        # if direction == 'Forward':
        #     ref_wavelength = wavelength
        # if direction == 'Reverse':
            # for i in range(len(aligned_rainbow.time.value)):
            #     aligned_rainbow.flux.value[:,i] = bintogrid(x=aligned_rainbow.wavelength.value, y=aligned_rainbow.flux.value[:,i], newx=ref_wavelength.value)['y']
            #     aligned_rainbow.wavelength = ref_wavelength
        
        'Trim the bad wavelengths from the wavelength and flux arrays'
        a = (wave_upper >= wavelength)
        b = (wavelength >= wave_lower)
        ok_wavelengths = (a == b) # trim the edges of each spectrum
        wave = wavelength[ok_wavelengths]    
        _flux = aligned_rainbow.flux[ok_wavelengths,:]
        unc = aligned_rainbow.uncertainty[ok_wavelengths, :]
        time = aligned_rainbow.time

        'Turn the wavelength-trimmed data into a Rainbow object and save to a file to use later'
        trimmed = Rainbow(wavelength=wave, time=aligned_rainbow.time, flux=_flux, uncertainty=unc)
        # trimmed.save(f"../../data/rainbows/{visit}_{direction}_trimmed_pacman_spec.rainbow.npy")

        # alter this so that the rainbow is the summed-and-normalized flux with relative uncertainties
        # (Scan combined)

        # trimmed.pcolormesh()
        'Calculate the white light curve'
        white_light_curve = np.nansum(trimmed.flux,axis=0)
        'Calculate the average spectrum'
        average_spec = trimmed.get_average_spectrum()
        average_spec_err = 0.005*average_spec #up to debate how to calculate this
        
        ''' Plot  '''
        # ax1.errorbar(trimmed.wavelength, average_spec, yerr=average_spec_err,label=f'{direction}')
        # ax2.scatter((time-t0)*24, white_light_curve, label=f'{direction}',s=2)

        for i,wavelength in enumerate(trimmed.wavelength.value):
            this_flux = trimmed.flux[i,:]
            normed_flux = this_flux / np.nanmedian(this_flux)
            trimmed.flux[i,:] = normed_flux*u.electron

        print(f'N_wavelengths={trimmed.wavelength.size}')

        plt.figure()

        if visit == 'F21':
            time_offset = 2459455 * u.day
        elif visit == 'S22':
            time_offset = 2459684 * u.day

        trimmed.time = trimmed.time - time_offset
        trimmed.pcolormesh()
        # plt.title(f'{visit} {direction} Trimmed')
        plt.xlabel(f'Time (+ {time_offset:.0f})',fontsize=13)
        plt.tick_params(axis='both', labelsize=13)
        plt.ylabel(r'Wavelength ($\mu$m)', fontsize=13)
        plt.savefig(f"../../figs/{visit}_{direction}_trimmed_normed_imshow.png",dpi=600)
        plt.show()

        # plt.title(f'{title}', fontsize=16)
    
        # plt.figure()
        # trimmed.plot()
        # plt.title(f'{visit} {direction} Trimmed')
        # plt.savefig(f"../figs/{visit}_{direction}_trimmed_speclcs.png",dpi=300)

    # ax1.legend()
    # ax1.set_title(f'Time-averaged {grism} spectrum')
    # ax1.set_xlabel(r'Wavelength [$\mu$m]')
    # ax1.set_ylabel(r'Flux (e$^-$')
    
    # ax2.axvspan(-1.75, 1.75, alpha=0.3,color='r',label='Dur = 3.5 Hr')
    # ax2.legend()
    # ax2.set_title(f'White light curve')
    # ax2.set_xlabel(r'Time from expected mid-transit (hr)')
    # # ax2.set_ylabel(r'Flux (e$^-$')
    
    # # plt.suptitle(f'{visit} Trimmed Data',size=20)
    # plt.savefig(f"../figs/{visit}_processed_data.png",dpi=300)

    

In [ ]:
for visit in ['S22','F21']:
    
    'Read in the trimmed rainbows from process_pacman_spectra.ipynb'
    data_fluxes = []
    spec_errs = []
    img_dates = []
    wave_arrays = []
    for direction in ['Forward','Reverse']:
        rainbow = read_rainbow(f"../../data/rainbows/{visit}_{direction}_trimmed_pacman_spec.rainbow.npy")
        img_dates.append(rainbow.time.value)
        wave_arrays.append(rainbow.wavelength.value)
        data_fluxes.append(rainbow.flux.value)
        spec_errs.append(rainbow.uncertainty.value)
    data_wavelength = np.nanmean(wave_arrays, axis=0)
    img_date = np.nanmean(img_dates,axis=0)
    data_flux = np.nansum(data_fluxes,axis=0)
    data_err = np.sqrt(spec_errs[0]**2 + spec_errs[1]**2) * u.electron

    scan_combined_rainbow = Rainbow(wavelength=data_wavelength*u.micron, time=img_date*u.day, flux=data_flux * u.electron, uncertainty=data_err)
    scan_combined_rainbow.pcolormesh()
    scan_combined_rainbow.save(f"../../data/rainbows/{visit}_scan-combined_trimmed_pacman_spec.rainbow.npy")

    relative_err =  np.sqrt(data_flux)/data_flux
    mean_data_flux = np.nanmean(data_flux, axis=1)
    normalized_data_flux = data_flux / mean_data_flux[:, np.newaxis]
    
    # for i in range(len(data_wavelength)):
    #     print(f'{visit} | Median relative uncertainty at {data_wavelength[i]:.4f} micron = {int(np.nanmedian(relative_err[i,:])*1e6)} ppm')
    #     plt.figure(figsize = (5,2))
    #     plt.title(f'{data_wavelength[i]:.4f} micron')
    #     plt.errorbar(img_date[20:], normalized_data_flux[i,20:],relative_err[i,20:],fmt='o',ms=1)